In [1]:
from pathlib import Path
import sys
import os
project_root = Path.cwd().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from Data.dataset import MVTecDataset
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from extractor import WideResNetFeatureExtractor
import tqdm
data_path = project_root / "Data" / "mvtec"
output_path = project_root / "memory_bank"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
sample_path = data_path / "bottle"
len(MVTecDataset(sample_path, split="train"))

In [ ]:
loader = DataLoader(MVTecDataset(sample_path, split="train"), batch_size=8)

In [ ]:
next(iter(loader))["image"].shape

In [ ]:
extractor = WideResNetFeatureExtractor().to(device)

In [ ]:
with torch.no_grad():
    embeddings = extractor(next(iter(loader))["image"].to(device))
embeddings.shape

In [ ]:
all_embeddings = []
with torch.no_grad():
    for img_batch in tqdm.tqdm(loader):
        embeddings = extractor(img_batch["image"].to(device))
        all_embeddings.append(embeddings.cpu())

In [ ]:
memory_bank = torch.cat(all_embeddings, dim=0)

print(memory_bank.shape)

In [ ]:
memory_bank = memory_bank.reshape(-1, memory_bank.shape[-1])

print(memory_bank.shape)

In [ ]:
torch.save(memory_bank.cpu(), f"{output_path}/memory_bank_bottle.pt")

In [ ]:
os.listdir(data_path)

In [ ]:
for sample in os.listdir(data_path):
    sample_path = data_path / sample
    dataloader = DataLoader(MVTecDataset(sample_path, split="train"), batch_size=8)
    result_path = f"{output_path}/memory_bank_{sample}.pt"
    all_embeddings = []
    if os.path.exists(result_path):
        print(f"Memory bank for {sample} already exists. Skipping...")
        continue
    extractor.eval()
    with torch.no_grad():
        for img_batch in tqdm.tqdm(dataloader):
            embeddings = extractor(img_batch["image"].to(device))
            all_embeddings.append(embeddings.cpu())
        
        memory_bank = torch.cat(all_embeddings, dim=0)
        memory_bank = memory_bank.reshape(-1, memory_bank.shape[-1])
        torch.save(memory_bank.cpu(), result_path)

In [ ]:
from memory_bank import create_memory_bank

In [ ]:
create_memory_bank(Dataset=MVTecDataset, Feature_Extractor=WideResNetFeatureExtractor, data_path=data_path, batch_size=8)

In [ ]:
bank = torch.load(f"{output_path}/memory_bank_bottle.pt")
print(bank.shape)
print(bank.dtype)

In [2]:
memory_bank = torch.load(f"{output_path}/memory_bank_bottle.pt")
memory_bank.shape

C:\Users\Kartik Wadhwa\AppData\Local\Temp\ipykernel_11008\2535349536.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  memory_bank = torch.load(f"{output_path}/memory_bank

torch.Size([214016, 1536])

In [7]:
int(len(memory_bank) * 0.01)

2140